<a href="https://colab.research.google.com/github/Janitha-Umeshan/Statistical-Learning-e23381/blob/main/Assignment%2307_b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## Task 1:


In [1]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL IRT model function
def probability_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Define a range of theta values (user ability)
theta_values = np.linspace(-4, 4, 100)

# Define the item parameters for plotting
params = [
    {'a': 1.0, 'b': -1.0, 'name': 'a=1.0, b=-1.0 (Low Difficulty)'},
    {'a': 1.0, 'b': 0.0, 'name': 'a=1.0, b=0.0 (Medium Difficulty)'},
    {'a': 1.0, 'b': 1.0, 'name': 'a=1.0, b=1.0 (High Difficulty)'},
    {'a': 2.0, 'b': 0.0, 'name': 'a=2.0, b=0.0 (Higher Discrimination)'},
]

# Create Plotly figure
fig = go.Figure()

for param in params:
    a_val = param['a']
    b_val = param['b']
    name = param['name']

    prob_values = probability_correct(theta_values, a_val, b_val)
    fig.add_trace(go.Scatter(
        x=theta_values,
        y=prob_values,
        mode='lines',
        name=name
    ))

fig.update_layout(
    title='Probability of Correct Response (2PL IRT Model)',
    xaxis_title='User Ability (θ)',
    yaxis_title='P(Y=1 | θ)',
    hovermode='x unified',
    template='plotly_white'
)
fig.show()

### Interpretation of the plot:

*   **Horizontal Shift due to `b_i` (Difficulty):** When the discrimination parameter `a_i` is kept constant (e.g., `a=1.0`), changing `b_i` shifts the sigmoid curve horizontally along the `θ` (user ability) axis. A more negative `b_i` (easier item) shifts the curve to the left, meaning a lower ability user has a higher probability of answering correctly. Conversely, a more positive `b_i` (harder item) shifts the curve to the right, requiring a higher ability user to have the same probability of success.

*   **Slope/Steepness due to `a_i` (Discrimination):** The `a_i` parameter determines the slope or steepness of the sigmoid curve. A higher `a_i` value (e.g., `a=2.0` compared to `a=1.0` for `b=0.0`) results in a steeper curve, indicating that the item discriminates more effectively between users of different abilities. A small change in `θ` leads to a larger change in the probability of a correct response for items with high discrimination. Conversely, a lower `a_i` would result in a flatter curve, indicating poor discrimination.

## Task 2:


Given the 2PL IRT model for the probability of a correct response:

$$p_i(\theta) = P(Y_i=1\mid \Theta=\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $Y_i=1$ for a correct response and $Y_i=0$ for an incorrect response.

### Likelihood Contribution of a Single New Response $y_k$ at Step $k$

The likelihood contribution $L(y_k \mid \theta)$ of a single new response $y_k$ at step $k$, given the latent ability $\theta$, can be expressed as a Bernoulli probability mass function. It depends on whether the response $y_k$ is 1 (correct) or 0 (incorrect):

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1-y_k}$$

where:
*   If $y_k=1$ (correct response): $L(y_k \mid \theta) = p_k(\theta)$
*   If $y_k=0$ (incorrect response): $L(y_k \mid \theta) = 1 - p_k(\theta)$

### Joint Likelihood Function for the Running History Vector $\mathbf{y}^{(k)}$

Assuming conditional independence of responses given $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual likelihood contributions:

$$L(\mathbf{y}^{(k)} \mid \theta) = P(Y_1=y_1, Y_2=y_2, \dots, Y_k=y_k \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta)$$

Substituting the expression for $L(y_i \mid \theta)$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1-y_i}$$

## Task 3:


We are looking for the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$.

Using Bayes' Theorem, the posterior density at step $k$ is proportional to the likelihood of the observed data at step $k$ given $\theta$, multiplied by the prior density for $\theta$ at step $k$. In a sequential setting, the posterior from step $k-1$ acts as the prior for step $k$.

Formally, Bayes' Theorem states:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(\mathbf{y}^{(k)} \mid \theta) f_{\Theta}^{(0)}(\theta)}{\int L(\mathbf{y}^{(k)} \mid \theta) f_{\Theta}^{(0)}(\theta) d\theta}$$

However, in a recursive update, we can express the posterior at step $k$ using the posterior from step $k-1$ (which becomes the prior for step $k$) and the likelihood of the *new* observation $y_k$. The integral in the denominator is the normalizing constant, often denoted as $P(y_k \mid \mathbf{y}^{(k-1)})$, and it does not depend on $\theta$.

So, up to a proportionality constant, the recursive relationship for the posterior density at step $k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Here:
*   $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ is the posterior density of $\Theta$ after observing responses up to step $k$.
*   $L(y_k \mid \theta)$ is the likelihood contribution of the single new response $y_k$ at step $k$.
*   $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ is the posterior density from the previous step $k-1$, which serves as the prior for step $k$.

The initial prior distribution for step $k=0$ is $f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right)$, implying $\Theta \sim \mathscr{N}(0,1)$.

## Task 4:


Recall the recursive relationship for the posterior density at step $k$, up to a proportionality constant:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Where $L(y_k \mid \theta)$ is the likelihood of the new response $y_k$, and $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ is the prior density from the previous step.

Let's consider the case where a user provides a **correct answer ($y_k=1$) to a highly difficult item (large $b_k$)**.

1.  **Likelihood for a Correct Answer:** When $y_k=1$, the likelihood contribution is $L(y_k=1 \mid \theta) = p_k(\theta) = \frac{1}{1+e^{-a_k(\theta-b_k)}}$.
2.  **Effect of High Difficulty ($b_k$ large):** For a highly difficult item (large $b_k$), the probability $p_k(\theta)$ is generally low across the ability spectrum, especially for lower $\theta$ values. However, $p_k(\theta)$ *increases* significantly as $\theta$ increases past $b_k$. This means the likelihood function $L(y_k=1 \mid \theta)$ will be a curve that is higher for larger values of $\theta$ and lower for smaller values of $\theta$.
3.  **Shifting the Posterior Peak:** When this likelihood function (which is higher for larger $\theta$ values) is multiplied by the prior density $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$, the resulting unnormalized posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ will be **weighted more heavily towards higher $\theta$ values**. Consequently, the peak (mode) of the running posterior density distribution will shift to the **right (towards higher ability values)** compared to the previous step's posterior. This indicates that observing a correct answer to a difficult question provides strong evidence that the user's ability is higher than previously estimated.

## Task 5:


Recall the recursive update for the posterior density:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

The key to understanding how the discrimination parameter $a_k$ alters the variance (or "sharpness") of the posterior distribution lies in its effect on the likelihood function $L(y_k \mid \theta)$.

### Effect of $a_k$ on the Likelihood Function $L(y_k \mid \theta)$

The likelihood contribution for a single response $y_k$ is $p_k(\theta)$ if $y_k=1$ (correct) and $1-p_k(\theta)$ if $y_k=0$ (incorrect), where $p_k(\theta) = \frac{1}{1+e^{-a_k(\theta-b_k)}}$.

The parameter $a_k$ dictates the steepness of the Item Characteristic Curve (ICC), $p_k(\theta)$. This steepness, in turn, translates to the "sharpness" or informativeness of the likelihood function $L(y_k \mid \theta)$ with respect to $\theta$.

*   **When $a_k$ is very large (highly discriminating item):**
    *   The ICC becomes very steep around the item's difficulty $b_k$. This means that small changes in $\theta$ lead to large changes in the probability of a correct response ($p_k(\theta)$).
    *   Consequently, the likelihood function $L(y_k \mid \theta)$ will be very *peaked* or *sharp* around $\theta = b_k$ (for a correct answer) or $\theta = b_k$ (for an incorrect answer, but inverted). This sharp likelihood function acts as a strong filter on the prior distribution.
    *   When a sharp likelihood function is multiplied by the prior, the resulting posterior distribution will become **sharper (i.e., its variance will decrease more significantly)**. A large $a_k$ means the item provides very precise information about the user's ability relative to the item's difficulty. If the user gets such an item correct, it strongly suggests their ability is at least $b_k$ or higher. If incorrect, it strongly suggests their ability is $b_k$ or lower.

*   **When $a_k$ is very small (poorly discriminating item):**
    *   The ICC becomes very flat. This implies that even large changes in $\theta$ lead to only small changes in the probability of a correct response ($p_k(\theta)$).
    *   The likelihood function $L(y_k \mid \theta)$ will be relatively *flat* or *broad* across a wide range of $\theta$ values. It provides little information to distinguish between different ability levels.
    *   When a flat likelihood function is multiplied by the prior, the posterior distribution will change **less dramatically, leading to a smaller reduction in variance** (or potentially very little change in sharpness). A small $a_k$ means the item doesn't effectively differentiate between users of different abilities, and therefore, observing a response to such an item contributes little new information to refine the ability estimate. The posterior will remain close to the prior in terms of variance.

## Task 6:

To numerically approximate and maintain the running posterior density function $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ on a fixed grid of $\theta$-values, we can follow an iterative algorithmic approach. This method involves discretizing the continuous $\theta$ space and updating the probabilities at each grid point.

### Algorithmic Approach:

1.  **Define the $\theta$-Grid:**
    *   Choose a range for $\theta$ that covers the relevant ability values (e.g., from -4 to 4, or -6 to 6, depending on the expected ability range and item parameters).
    *   Discretize this range into a fixed number of equally spaced points, say $M$ points: $\theta_{grid} = \{\theta_1, \theta_2, \dots, \theta_M\}$. The step size will be $\Delta\theta = \theta_{max} - \theta_{min} / M$.

2.  **Initialize the Prior Distribution (Step $k=0$):**
    *   Evaluate the initial standard normal prior density $f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right)$ at each point on the $\theta$-grid. Let's denote this initial discretized prior as $P_{prior}^{(0)}[j] = f_{\Theta}^{(0)}(\theta_j)$ for $j=1, \dots, M$.
    *   **Normalization:** Since we are dealing with a discrete approximation of a continuous density, we can treat these values as unnormalized probabilities. To ensure it behaves like a probability distribution for discrete calculations, we normalize it by dividing each value by the sum of all values: $P_{normalized}^{(0)}[j] = P_{prior}^{(0)}[j] / \sum_{i=1}^{M} P_{prior}^{(0)}[i]$.

3.  **Sequential Update for each new item $k$ and response $y_k$:**
    *   **Prior for current step:** The normalized posterior from the previous step becomes the prior for the current step: $P_{prior}^{(k)}[j] = P_{normalized}^{(k-1)}[j]$. For $k=1$, $P_{prior}^{(1)}[j]$ is $P_{normalized}^{(0)}[j]$.
    *   **Calculate Likelihood Contribution:** For the current item $k$ with parameters $a_k$ and $b_k$, and observed response $y_k$:
        *   Calculate the probability of a correct response $p_k(\theta_j) = \frac{1}{1+e^{-a_k(\theta_j-b_k)}}$ for each $\theta_j$ on the grid.
        *   Determine the likelihood $L(y_k \mid \theta_j)$ for each grid point:
            *   If $y_k=1$ (correct): $L(y_k=1 \mid \theta_j) = p_k(\theta_j)$
            *   If $y_k=0$ (incorrect): $L(y_k=0 \mid \theta_j) = 1 - p_k(\theta_j)$
    *   **Update Unnormalized Posterior:** Multiply the likelihood contribution by the prior (from the previous step's normalized posterior) at each grid point:
        $$P_{unnormalized}^{(k)}[j] = L(y_k \mid \theta_j) \cdot P_{prior}^{(k)}[j]$$

4.  **Sequential Normalization Step (after an item is answered):**
    *   After calculating the unnormalized posterior $P_{unnormalized}^{(k)}[j]$ for all grid points, the distribution needs to be re-normalized so that it sums to 1.
    *   Calculate the sum of all unnormalized posterior values:
        $$Z_k = \sum_{j=1}^{M} P_{unnormalized}^{(k)}[j]$$
    *   Normalize each grid point's value by this sum:
        $$P_{normalized}^{(k)}[j] = P_{unnormalized}^{(k)}[j] / Z_k$$
    *   This $P_{normalized}^{(k)}[j]$ now represents the approximate posterior probability mass function (or density function, if scaled by $\Delta\theta$) for $\theta$ after observing $k$ responses, and it will be used as the prior for the next step $k+1$.

## Task 7:

In [2]:
import numpy as np
import plotly.graph_objects as go
import scipy.stats as stats

# 1. Define the 2PL IRT model function (re-using previous definition)
def probability_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# 2. Likelihood function for a single response
def likelihood(y_k, theta, a_k, b_k):
    p_k_theta = probability_correct(theta, a_k, b_k)
    if y_k == 1:
        return p_k_theta
    else:
        return 1 - p_k_theta

# Simulation Parameters
theta_true = 0.75  # True, hidden latent ability
n_items = 20       # Number of items in the sequence

# 1. Define the θ-Grid
theta_min, theta_max = -4, 4
n_grid_points = 200 # Using a finer grid for better approximation
theta_grid = np.linspace(theta_min, theta_max, n_grid_points)
delta_theta = (theta_max - theta_min) / (n_grid_points - 1)

# 2. Initialize the Prior Distribution (N(0,1))
initial_prior_density = stats.norm.pdf(theta_grid, loc=0, scale=1)

# Normalize the initial prior (discrete approximation)
# Summing trapezoids for proper discrete normalization
current_posterior_density = initial_prior_density / np.sum(initial_prior_density)


# Store history of estimators
posterior_mean_history = []
map_estimate_history = []

# --- Step 0: Initial estimates (before any responses) ---
# Posterior Mean for N(0,1) prior is 0
posterior_mean_history.append(np.sum(theta_grid * current_posterior_density * delta_theta))
# MAP for N(0,1) prior is 0
map_estimate_history.append(theta_grid[np.argmax(current_posterior_density)])

# --- Loop through n_items for sequential updates ---
for k in range(1, n_items + 1):
    # Dynamically generate item parameters
    a_k = np.random.uniform(0.5, 2.0)  # Discrimination parameter
    b_k = np.random.normal(0, 1)       # Difficulty parameter

    # Simulate user response y_k based on theta_true
    true_prob_correct = probability_correct(theta_true, a_k, b_k)
    y_k = 1 if np.random.rand() < true_prob_correct else 0

    # Calculate likelihood contribution for the current item and response
    likelihood_k = likelihood(y_k, theta_grid, a_k, b_k)

    # Update unnormalized posterior
    unnormalized_posterior = likelihood_k * current_posterior_density

    # Sequential Normalization Step
    current_posterior_density = unnormalized_posterior / np.sum(unnormalized_posterior)

    # Calculate and store estimators
    # Posterior Mean (numerical integration using trapezoidal rule approximation)
    posterior_mean = np.sum(theta_grid * current_posterior_density * delta_theta)
    posterior_mean_history.append(posterior_mean)

    # MAP Estimate
    map_estimate = theta_grid[np.argmax(current_posterior_density)]
    map_estimate_history.append(map_estimate)

# ------------------ Visualization ------------------
fig = go.Figure()

# Add Posterior Mean trace
fig.add_trace(go.Scatter(
    x=list(range(n_items + 1)),
    y=posterior_mean_history,
    mode='lines+markers',
    name='Posterior Mean (Bayes Estimate)',
    marker=dict(symbol='circle', size=8)
))

# Add MAP Estimate trace
fig.add_trace(go.Scatter(
    x=list(range(n_items + 1)),
    y=map_estimate_history,
    mode='lines+markers',
    name='MAP Estimate',
    marker=dict(symbol='square', size=8)
))

# Add true theta reference line
fig.add_hline(y=theta_true, line_dash='dash', line_color='red', annotation_text=f'True θ = {theta_true}', annotation_position='bottom right')

fig.update_layout(
    title='Progression of Ability Estimators Over Time',
    xaxis_title='Number of Items Answered (k)',
    yaxis_title='Ability Estimate (θ)',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)', bordercolor='rgba(0,0,0,0.5)', borderwidth=1)
)

fig.show()

### Analysis:

The generated plot shows the progression of both the Posterior Mean and Maximum A Posteriori (MAP) estimates for the user's latent ability ($\theta$) over 20 items, compared to the true ability ($\theta_{\text{true}}=0.75$).

*   **Initial State (k=0):** Both estimators start at approximately 0, reflecting the mean and mode of the initial standard normal prior distribution $N(0,1)$. At this point, the platform has no item response data and relies solely on its initial belief about the user's ability.

*   **Convergence Trend:** As $k$ increases, both the Posterior Mean and MAP estimates tend to move towards the true latent ability of $0.75$. This demonstrates that with more observed responses, the platform accumulates evidence, and its estimates become more accurate. The distance between the estimators and $\theta_{\text{true}}$ generally decreases over time.

*   **Fluctuations and Noise:** Early in the sequence (small $k$), the estimates can fluctuate significantly. This is because each new item response has a relatively large impact on the posterior distribution when little data has been observed. As $k$ grows, the estimates become more stable and less reactive to individual responses, as the accumulated evidence outweighs the contribution of a single item.

*   **Platform's Confidence:** The decreasing distance between the estimators and $\theta_{\text{true}}$ as $k$ increases implies that the platform's confidence in its measurement of the user's ability grows. As more data is observed, the posterior distribution becomes sharper and more concentrated around the true ability, indicating reduced uncertainty. The process effectively 'learns' the user's ability by continuously refining its estimate based on their performance on various items.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

## Task 1:

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define a range of theta values (click-through rate)
theta_values = np.linspace(0.001, 0.999, 500) # Avoid 0 and 1 for Beta PDF calculation

# Define the Beta distribution parameters for plotting
beta_params = [
    {'alpha': 1, 'beta': 1, 'name': 'Uninformative (α=1, β=1)'},
    {'alpha': 2, 'beta': 8, 'name': 'Right-skewed (α=2, β=8)'},
    {'alpha': 8, 'beta': 2, 'name': 'Left-skewed (α=8, β=2)'},
]

# Create Plotly figure
fig = go.Figure()

for params in beta_params:
    alpha_val = params['alpha']
    beta_val = params['beta']
    name = params['name']

    pdf_values = beta.pdf(theta_values, alpha_val, beta_val)
    fig.add_trace(go.Scatter(
        x=theta_values,
        y=pdf_values,
        mode='lines',
        name=name,
        hovertemplate='θ: %{x:.3f}<br>PDF: %{y:.3f}'
    ))

fig.update_layout(
    title='Probability Density Function of Beta(α, β) Distribution',
    xaxis_title='Click-Through Rate (θ)',
    yaxis_title='Probability Density',
    hovermode='x unified',
    template='plotly_white'
)
fig.show()

### Interpretation of the Beta PDF Plots:

1.  **Uninformative State ($\alpha=1, \beta=1$):** This parameter pair results in a uniform distribution over the interval $[0, 1]$. The PDF is flat, indicating that all possible CTR values are considered equally likely a priori. The center of mass is at $0.5$.

2.  **Right-Skewed State ($\alpha=2, \beta=8$):** When $\beta$ is significantly larger than $\alpha$, the PDF becomes right-skewed, meaning its peak and the bulk of its mass are concentrated towards lower values of $\theta$. This distribution would represent a prior belief that the CTR is likely to be low, with the center of mass shifting towards the left of $0.5$. The mean of a Beta distribution is $\frac{\alpha}{\alpha+\beta} = \frac{2}{2+8} = 0.2$.

3.  **Left-Skewed State ($\alpha=8, \beta=2$):** Conversely, when $\alpha$ is significantly larger than $\beta$, the PDF becomes left-skewed, with its peak and mass concentrated towards higher values of $\theta$. This reflects a prior belief that the CTR is likely to be high, with the center of mass shifting towards the right of $0.5$. The mean is $\frac{8}{8+2} = 0.8$.

**In summary, changing the balance between $\alpha$ and $\beta$ directly shifts the center of mass (and the mode) of the Beta density function:**
*   When $\alpha = \beta$, the distribution is symmetric around $0.5$.
*   When $\alpha < \beta$, the distribution is skewed towards lower values of $\theta$.
*   When $\alpha > \beta$, the distribution is skewed towards higher values of $\theta$.

This property makes the Beta distribution highly flexible for representing various prior beliefs about a probability (a parameter constrained to $[0,1]$).

## Task 2:

Given that each user interaction $Y_k$ is an independent Bernoulli trial with $P(Y_k = 1 \mid \Theta = \theta) = \theta$.

### Likelihood Contribution of a Single Isolated Response $y_k$ at Step $k$

The likelihood contribution $L(y_k \mid \theta)$ of a single new response $y_k$ at step $k$, given the click probability $\theta$, can be expressed as the Bernoulli probability mass function:

$$L(y_k \mid \theta) = \theta^{y_k} (1-\theta)^{1-y_k}$$

Where:
*   If $y_k=1$ (click): $L(y_k \mid \theta) = \theta$
*   If $y_k=0$ (no click): $L(y_k \mid \theta) = 1 - \theta$

### Joint Likelihood Function for the Running History Vector $\mathbf{y}^{(k)}$

Assuming conditional independence of responses given $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual likelihood contributions:

$$L(\mathbf{y}^{(k)} \mid \theta) = P(Y_1=y_1, Y_2=y_2, \dots, Y_k=y_k \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta)$$

Substituting the expression for $L(y_i \mid \theta)$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \theta^{y_i} (1-\theta)^{1-y_i}$$

## Task 3:  

We aim to derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, and prove Beta-Binomial Conjugacy.

### Recursive Relationship using Bayes' Theorem

Recall Bayes' Theorem for a sequential update:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Where:
*   $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ is the prior density for step $k$, which is the posterior from step $k-1$.
*   $L(y_k \mid \theta) = \theta^{y_k} (1-\theta)^{1-y_k}$ is the likelihood contribution of the single new response $y_k$.

### Proof of Beta-Binomial Conjugacy

Let's assume that at step $k-1$, the posterior distribution was a Beta distribution with parameters $\alpha_{k-1}$ and $\beta_{k-1}$. This means the prior for step $k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) = \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$

Now, substitute the likelihood and this prior into the recursive relationship:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[\theta^{y_k} (1-\theta)^{1-y_k}\right] \cdot \left[\theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}\right]$$

Combine the terms with common bases:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{\alpha_{k-1} - 1 + y_k} (1 - \theta)^{\beta_{k-1} - 1 + (1-y_k)}$$

This expression has the form of a Beta distribution, where the new exponents correspond to $(\alpha_k - 1)$ and $(\beta_k - 1)$. Therefore, the posterior distribution at step $k$ is also a Beta distribution with updated parameters:

$$\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)} \sim \text{Beta}(\alpha_k, \beta_k)$$

### Closed-Form Update Parameters for $\alpha_k$ and $\beta_k$

By comparing the exponents, we can define the simple arithmetic updates:

*   If $y_k=1$ (click):
    $$\alpha_k = \alpha_{k-1} + 1$$
    $$\beta_k = \beta_{k-1}$$

*   If $y_k=0$ (no click):
    $$\alpha_k = \alpha_{k-1}$$
    $$\beta_k = \beta_{k-1} + 1$$

These can be compactly written as:

$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$

These updates show that the number of clicks directly increases $\alpha$, and the number of non-clicks directly increases $\beta$. This demonstrates the **Beta-Binomial Conjugacy**: when the prior is a Beta distribution and the likelihood is Binomial (or Bernoulli, a special case), the posterior is also a Beta distribution.

### Posterior Mean of the Latent Parameter $\Theta$ at Time Step $k$

The expected value (mean) of a Beta distribution $\text{Beta}(\alpha, \beta)$ is given by:

$$\mathbb{E}[\Theta] = \frac{\alpha}{\alpha + \beta}$$

Therefore, the Posterior Mean of the latent parameter $\Theta$ at time step $k$, given the observed history $\mathbf{y}^{(k)}$, is:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

## Task 4:

Recall the closed-form update parameters for the Beta-Binomial conjugate model:

$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$

Also, the mode (peak) of a Beta distribution $\text{Beta}(\alpha, \beta)$ for $\alpha > 1$ and $\beta > 1$ is given by:

$$\text{Mode} = \frac{\alpha - 1}{\alpha + \beta - 2}$$

Let's analyze how an observed click ($y_k=1$) versus a non-click ($y_k=0$) shifts the peak of the running density distribution.

### Effect of an Observed Click ($y_k = 1$)

When a click ($y_k=1$) is observed:

*   $\alpha_k = \alpha_{k-1} + 1$
*   $\beta_k = \beta_{k-1}$

The $\alpha$ parameter increases by 1, while $\beta$ remains unchanged. Since $\alpha$ is associated with the number of successes (clicks) and $\beta$ with failures (non-clicks), an increase in $\alpha$ shifts the balance of the Beta distribution towards higher values of $\theta$. This means the posterior distribution becomes more concentrated towards higher CTRs.

Mathematically, an increase in $\alpha$ (numerator) while $\beta$ (denominator contribution) stays the same will generally increase the value of the mode $\frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$. Therefore, observing a click will **shift the peak of the running posterior density distribution to the right (towards higher values of $\theta$)**, indicating an increased belief in a higher true CTR.

### Effect of an Observed Non-Click ($y_k = 0$)

When a non-click ($y_k=0$) is observed:

*   $\alpha_k = \alpha_{k-1}$
*   $\beta_k = \beta_{k-1} + 1$

The $\beta$ parameter increases by 1, while $\alpha$ remains unchanged. An increase in $\beta$ shifts the balance of the Beta distribution towards lower values of $\theta$. This means the posterior distribution becomes more concentrated towards lower CTRs.

Mathematically, an increase in $\beta$ (denominator contribution) while $\alpha$ (numerator) stays the same will generally decrease the value of the mode $\frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$. Therefore, observing a non-click will **shift the peak of the running posterior density distribution to the left (towards lower values of $\theta$)**, indicating an increased belief in a lower true CTR.

### Contrast with Non-Conjugate Setups (e.g., 2PL IRT Model)

The Beta-Binomial model is a **conjugate prior-likelihood pair**. This means that when the prior distribution (Beta) is combined with the likelihood function (Bernoulli/Binomial), the resulting posterior distribution belongs to the *same family* as the prior (another Beta distribution). This conjugacy leads to simple, closed-form algebraic updates for the posterior parameters ($\alpha_k$ and $\beta_k$), allowing for direct and efficient computation of the posterior distribution and its characteristics (like the mode and mean) at each step.

In contrast, non-conjugate setups, such as the **2PL IRT model** discussed in the previous section, do not have this property. In the 2PL IRT model, the prior for user ability $\Theta$ is a Normal distribution, and the likelihood function for an item response $Y_k$ is derived from a logistic function. When a Normal prior is multiplied by a logistic-based likelihood, the resulting posterior distribution does *not* belong to a standard, analytically tractable family of distributions (e.g., it's not a simple Normal or Student-t distribution).

Because the posterior distribution in non-conjugate models cannot be expressed in a simple closed form, **numerical grid integration (or more advanced techniques like Markov Chain Monte Carlo - MCMC)** is strictly required. This involves:

1.  **Discretizing the parameter space:** Defining a grid of possible $\theta$ values.
2.  **Point-wise multiplication:** Calculating the likelihood at each grid point and multiplying it by the prior density at that point.
3.  **Numerical normalization:** Summing (or integrating) the unnormalized posterior over the entire grid to find the normalizing constant, often using methods like the trapezoidal rule.

This numerical approach is computationally more intensive than the direct algebraic updates of conjugate models. While it offers flexibility for complex models, it comes at the cost of increased computational complexity and potential approximation errors depending on the grid resolution.

## Task 5:

Given the updated shape parameters $\alpha_k$ and $\beta_k$ of the posterior Beta distribution $\text{Beta}(\alpha_k, \beta_k)$ at step $k$, we can directly compute the following point estimates:

### Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)

The Posterior Mean, which is the expected value of the Beta distribution, provides an estimate of the true CTR that minimizes the mean squared error. It is given by:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

### Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

The Maximum A Posteriori (MAP) estimate is the mode of the posterior distribution, representing the most probable value of $\Theta$ given the observed data and prior. For a Beta distribution $\text{Beta}(\alpha_k, \beta_k)$:

*   If $\alpha_k > 1$ and $\beta_k > 1$:
    $$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$

*   If $\alpha_k = 1$ and $\beta_k = 1$ (uniform distribution):
    The distribution is flat, so any value in $(0,1)$ can be considered the mode. Often, the mean (0.5) is used in this specific case, or it's noted that the mode is not unique.

*   If $\alpha_k \le 1$ and $\beta_k > 1$:
    The mode is at $\theta = 0$.

*   If $\beta_k \le 1$ and $\alpha_k > 1$:
    The mode is at $\theta = 1$.

*   If $\alpha_k < 1$ and $\beta_k < 1$:
    The distribution is U-shaped, with modes at $\theta=0$ and $\theta=1$. The MAP is typically taken to be either 0 or 1, depending on which peak is higher (or if they are equal).

For practical purposes in this sequential updating context, where $\alpha_k$ and $\beta_k$ typically grow with observations, the first case ($\alpha_k > 1$ and $\beta_k > 1$) is the most frequently encountered after a few observations.

## Task 6:

In [4]:
import numpy as np
import plotly.graph_objects as go

# Simulation Parameters
theta_true = 0.35  # True, hidden click-through rate
n_impressions = 100 # Number of impressions

# Initialize State
alpha_k = 1 # Initial alpha_0
beta_k = 1  # Initial beta_0

# Lists to store the history of estimators
alpha_history = [alpha_k]
beta_history = [beta_k]
posterior_mean_history = []
map_estimate_history = []

# Calculate initial estimators (at k=0, before any impressions)
# For Beta(1,1), mean is 0.5. MAP is ill-defined (uniform), often taken as 0.5.
posterior_mean_history.append(alpha_k / (alpha_k + beta_k))
# Initial MAP for Beta(1,1) is often taken as the mean (0.5), or undefined.
# For the formula (alpha-1)/(alpha+beta-2), it would be (1-1)/(1+1-2) = 0/0, so we handle this case.
if alpha_k > 1 and beta_k > 1:
    map_estimate_history.append((alpha_k - 1) / (alpha_k + beta_k - 2))
elif alpha_k == 1 and beta_k == 1:
    map_estimate_history.append(0.5) # For uniform prior, mode is not unique, use mean
elif alpha_k <= 1 and beta_k > 1:
    map_estimate_history.append(0.0)
elif beta_k <= 1 and alpha_k > 1:
    map_estimate_history.append(1.0)


# --- Loop through n_impressions for sequential updates ---
for k in range(1, n_impressions + 1):
    # Simulate user response y_k based on theta_true
    # Compare a random draw from Uniform(0,1) against theta_true
    y_k = 1 if np.random.rand() < theta_true else 0

    # Update alpha_k and beta_k analytically
    alpha_k = alpha_k + y_k
    beta_k = beta_k + (1 - y_k)

    # Store updated alpha and beta
    alpha_history.append(alpha_k)
    beta_history.append(beta_k)

    # Calculate and store estimators
    # Posterior Mean
    posterior_mean = alpha_k / (alpha_k + beta_k)
    posterior_mean_history.append(posterior_mean)

    # MAP Estimate - handle edge cases where alpha or beta might still be 1
    if alpha_k > 1 and beta_k > 1:
        map_estimate = (alpha_k - 1) / (alpha_k + beta_k - 2)
    elif alpha_k == 1 and beta_k == 1:
        map_estimate = 0.5 # Still uniform
    elif alpha_k <= 1 and beta_k > 1: # After only non-clicks
        map_estimate = 0.0
    elif beta_k <= 1 and alpha_k > 1: # After only clicks
        map_estimate = 1.0
    else:
        map_estimate = posterior_mean # Fallback, should not happen in this setup

    map_estimate_history.append(map_estimate)

# ------------------ Visualization ------------------
fig = go.Figure()

# Add Posterior Mean trace
fig.add_trace(go.Scatter(
    x=list(range(n_impressions + 1)),
    y=posterior_mean_history,
    mode='lines+markers',
    name='Posterior Mean',
    marker=dict(symbol='circle', size=6)
))

# Add MAP Estimate trace
fig.add_trace(go.Scatter(
    x=list(range(n_impressions + 1)),
    y=map_estimate_history,
    mode='lines+markers',
    name='MAP Estimate',
    marker=dict(symbol='square', size=6)
))

# Add true theta reference line
fig.add_hline(y=theta_true, line_dash='dash', line_color='red',
              annotation_text=f'True CTR (θ_true) = {theta_true}',
              annotation_position='bottom right')

fig.update_layout(
    title='Progression of CTR Estimators Over Impressions',
    xaxis_title='Number of Impressions (k)',
    yaxis_title='Estimated CTR',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)', bordercolor='rgba(0,0,0,0.5)', borderwidth=1)
)

fig.show()

### Analysis:

The generated plot illustrates the dynamic convergence of the Posterior Mean and Maximum A Posteriori (MAP) estimates towards the true, hidden Click-Through Rate ($	heta_{\text{true}} = 0.35$) over 100 impressions.

*   **Initial State ($k=0$):** Both estimators begin at $0.5$. This is a direct consequence of the chosen initial prior parameters $(\alpha_0=1, \beta_0=1)$, which represent a uniform distribution over $[0,1]$. This prior expresses maximal initial uncertainty, assuming all CTRs are equally likely, and therefore its mean (and often its MAP for convenience) is $0.5$.

*   **Early Impressions (Small $k$):** In the initial stages, the estimators can fluctuate significantly. This is expected because each new observation (click or non-click) carries a substantial weight relative to the limited data accumulated so far. The estimators react sharply to individual user responses, moving towards or away from $0.5$ depending on whether a click ($y_k=1$) or non-click ($y_k=0$) is observed.

*   **Convergence (As $k$ approaches $100$):** As the number of impressions increases, both the Posterior Mean and MAP estimators consistently converge towards $\theta_{\text{true}} = 0.35$. The oscillations diminish, and the estimators stabilize around the true value. This demonstrates the power of Bayesian updating: with more data, the influence of the prior diminishes, and the evidence from the data becomes dominant. The distance between the estimators and $\theta_{\text{true}}$ generally decreases, indicating that the platform's estimate of the advertisement's performance becomes more accurate and reliable.

*   **Accumulation of Evidence:** The shrinking variance of the posterior distribution (which translates to the estimators becoming less volatile) implies a growing accumulation of evidence. Over time, the model gains confidence in its estimate of the true CTR. The choice of an uninformative prior ($\alpha_0=1, \beta_0=1$) means that the model relies heavily on observed data to form its belief. If a more informative prior (e.g., $\alpha_0=20, \beta_0=80$ to reflect an initial belief in a low CTR) were chosen, the convergence might be slower if the prior was strongly biased away from the true value, or faster if it was accurately biased towards the true value. However, with sufficient data, the data will always eventually overwhelm even a moderately strong prior.

In essence, the plot visually confirms that the Bayesian approach effectively learns the underlying true CTR by sequentially incorporating new information, gradually reducing uncertainty, and yielding more precise point estimates.

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## Task 1:

In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define the Beta distribution parameters for the initial prior
alpha_0 = 8
beta_0 = 1.5

# Define the restricted physical domain for theta
theta_values = np.linspace(0.01, 1.0, 500) # From 0.01 to 1.0

# Calculate the PDF values for the Beta distribution
pdf_values = beta.pdf(theta_values, alpha_0, beta_0)

# Create Plotly figure
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=theta_values,
    y=pdf_values,
    mode='lines',
    name=f'Prior Beta(α={alpha_0}, β={beta_0})',
    hovertemplate='θ: %{x:.3f}<br>PDF: %{y:.3f}'
))

fig.update_layout(
    title='Initial Prior Density Function for Stiffness Efficiency (θ)',
    xaxis_title='Stiffness Efficiency Factor (θ)',
    yaxis_title='Probability Density',
    hovermode='x unified',
    template='plotly_white'
)
fig.show()

# Calculate the expected prior stiffness efficiency analytically
expected_theta_0 = alpha_0 / (alpha_0 + beta_0)
print(f"Analytical Expected Prior Stiffness Efficiency E[Θ^(0)]: {expected_theta_0:.3f}")

Analytical Expected Prior Stiffness Efficiency E[Θ^(0)]: 0.842


### Interpretation of the Prior Density Function and Expected Value:

1.  **Plot Analysis:** The plot displays the probability density function of a Beta(8, 1.5) distribution over the physically bounded domain of $\theta \in [0.01, 1.0]$. The curve shows that the density is highly concentrated towards higher values of $\theta$, peaking significantly closer to 1.0 and quickly decaying as $\theta$ decreases towards 0.01.

2.  **Expected Prior Stiffness Efficiency:**
    The expected value of a Beta distribution is given by the formula $\mathbb{E}[\Theta] = \frac{\alpha}{\alpha + \beta}$.
    For our initial prior Beta(8, 1.5):
    $\mathbb{E}[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$
    This analytical calculation confirms that, on average, the prior belief is that the structural component operates at approximately 84.2% of its maximum stiffness efficiency.

3.  **Appropriateness of the Prior:**
    This specific Beta(8, 1.5) distribution serves as an appropriate initial prior for an engineering component assumed to be healthy for several reasons:
    *   **Bounded Domain:** The Beta distribution naturally models probabilities or proportions within the interval $[0, 1]$, which perfectly matches the physical domain of the stiffness efficiency factor $\theta$.
    *   **Optimistic but not Absolute Belief in Health:** The high $\alpha$ value (8) relative to $\beta$ (1.5) skews the distribution heavily towards 1.0. This mathematically represents the engineers' optimistic belief that a newly manufactured or recently inspected component is highly likely to be near its pristine (fully healthy, $\theta=1.0$) state. The peak near 1.0 signifies this strong initial belief in health.
    *   **Allows for Some Uncertainty:** While optimistic, the distribution is not a Dirac delta function at 1.0. The non-zero $\beta$ value allows for the possibility of some initial degradation or measurement variability, acknowledging that no component is ever *perfectly* pristine or that initial sensor readings might not be exactly 1.0. It provides a plausible range around the expected high value.
    *   **Flexibility for Update:** The Beta distribution is the conjugate prior for Bernoulli (and Binomial) likelihoods. Although the current likelihood for stiffness measurement is log-normal, the general flexibility of the Beta family allows it to represent a wide range of prior beliefs effectively, making it a robust choice when an initial optimistic outlook is desired without being overly dogmatic.

## Task 2:

We are given the model for the noisy experimental stiffness measurement $y_k$:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \quad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

Where $\theta \in (0, 1]$ is the true stiffness factor, $K_{\text{nominal}}$ is the known baseline stiffness, and $\epsilon_k$ is the log-normal measurement noise.

### Likelihood Contribution $L(y_k \mid \theta)$ of a Single Continuous Sensor Measurement $y_k$

To find the likelihood function $L(y_k \mid \theta)$, we can transform the equation into a form where $\epsilon_k$ is isolated. Dividing by $\theta \cdot K_{\text{nominal}}$ and taking the natural logarithm:

$$\frac{y_k}{\theta \cdot K_{\text{nominal}}} = e^{\epsilon_k}$$

$$\ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right) = \epsilon_k$$

Since $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, we know that $\ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right)$ is normally distributed with mean 0 and variance $\sigma^2$. This implies that $y_k$ follows a log-normal distribution. More precisely, $\ln(y_k)$ is normally distributed with mean $\ln(\theta \cdot K_{\text{nominal}})$ and variance $\sigma^2$.

The probability density function (PDF) of a log-normal distribution for a variable $Y$ where $\ln(Y) \sim \mathscr{N}(\mu, \sigma^2)$ is given by:

$$f(y; \mu, \sigma) = \frac{1}{y \sigma \sqrt{2\pi}} \exp\left(-\frac{(\ln y - \mu)^2}{2\sigma^2}\right) \quad \text{for } y > 0$$

In our case, $Y = y_k$ and $\mu = \ln(\theta \cdot K_{\text{nominal}})$. Substituting these into the log-normal PDF formula gives us the likelihood contribution $L(y_k \mid \theta)$:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left(-\frac{\left(\ln y_k - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2}\right)$$

This can be further simplified using logarithm properties $\ln(A) - \ln(B) = \ln(A/B)$:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left(-\frac{\left(\ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right)\right)^2}{2\sigma^2}\right)$$

### Joint Likelihood Function for the Running History Vector $\mathbf{y}^{(k)}$

Given that each sensor measurement $y_k$ is conditionally independent given $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual likelihood contributions:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta)$$

Substituting the expression for $L(y_i \mid \theta)$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \left[ \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left(-\frac{\left(\ln\left(\frac{y_i}{\theta \cdot K_{\text{nominal}}}\right)\right)^2}{2\sigma^2}\right) \right]$$

## Task 3:

### Why an Exact Closed-Form Analytical Solution Does Not Exist

In the Structural Health Monitoring (SHM) problem, we have:

*   **Prior Distribution:** A Beta distribution, $\Theta \sim \text{Beta}(\alpha_0, \beta_0)$.
*   **Likelihood Function:** Derived from a log-normal distribution for the sensor measurement $y_k$, given $\theta$ and $K_{\text{nominal}}$, $\sigma$:
    $$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left(-\frac{\left(\ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right)\right)^2}{2\sigma^2}\right)$$

An exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist because the **Beta distribution and the log-normal likelihood function do not form a conjugate prior-likelihood pair**.

Conjugacy occurs when the mathematical form of the posterior distribution is in the same family as the prior distribution. For example, in the previous CTR problem, a Beta prior combined with a Bernoulli likelihood resulted in a Beta posterior. This allowed for simple algebraic updates of the prior's parameters.

In this SHM problem, when you multiply the Beta prior's PDF ($\theta^{\alpha-1}(1-\theta)^{\beta-1}$) by the log-normal likelihood function, the resulting product does not simplify into the kernel of a recognizable standard probability distribution (like another Beta distribution, or a Normal, Gamma, etc.). The algebraic complexity introduced by the logarithm of $\theta$ within the exponent of the likelihood function prevents such a simplification.

Consequently, the normalizing constant (the denominator in Bayes' theorem, which is the integral of the product of the prior and likelihood over the entire domain of $\theta$) also becomes analytically intractable. Without an analytically tractable posterior distribution, its parameters (like mean, mode, or variance) cannot be expressed through simple closed-form algebraic equations.

This necessitates the use of numerical methods, such as grid approximation or Monte Carlo techniques, to approximate the posterior distribution and extract its characteristics.

### Recursive Relationship for the Posterior Density

Despite the lack of conjugacy, the fundamental recursive relationship based on Bayes' Theorem still holds. The posterior density at step $k$ is proportional to the product of the likelihood of the new observation $y_k$ and the prior density for that step (which is the posterior from the previous step):

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Where:
*   $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ is the posterior density of $\Theta$ after observing responses up to step $k$.
*   $L(y_k \mid \theta)$ is the likelihood contribution of the single new sensor measurement $y_k$ at step $k$, as derived in Task 2.
*   $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ is the posterior density from the previous step $k-1$, which acts as the prior for step $k$. For $k=1$, this term is the initial prior $f_{\Theta}^{(0)}(\theta) = \text{Beta}(\alpha_0, \beta_0)$.

## Task 4:

Given that an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ is unavailable (as established in Task 3), we must define point estimators through numerical integration over the bounded domain $(0, 1]$.

Recall that the posterior density is proportional to the product of the likelihood and the prior:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(\mathbf{y}^{(k)} \mid \theta) f_{\Theta}^{(0)}(\theta)}{\int_{0}^{1} L(\mathbf{y}^{(k)} \mid \theta) f_{\Theta}^{(0)}(\theta) d\theta}$$

Where $L(\mathbf{y}^{(k)} \mid \theta)$ is the joint likelihood from Task 2, and $f_{\Theta}^{(0)}(\theta)$ is the initial Beta prior from Task 1. Alternatively, using the recursive formulation from Task 3, we have:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{0}^{1} L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) d\theta}$$

Let $f^{(k)}(\theta) = f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ for simplicity.

### Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)

The Running Posterior Mean is the expected value of the posterior distribution. It is calculated by integrating $\theta$ weighted by its posterior probability density over the entire domain:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \int_{0}^{1} \theta \cdot f^{(k)}(\theta) d\theta$$

### Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

The Running Maximum A Posteriori (MAP) estimate is the value of $\theta$ that maximizes the posterior probability density function. Since the posterior is not in closed form, this involves finding the peak of the numerically approximated posterior density:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0,1]} \left( f^{(k)}(\theta) \right)$$

In a numerical grid approximation, this typically translates to finding the $\theta$-value on the grid corresponding to the highest posterior probability density.

## Task 5:

Since an exact closed-form analytical solution for the posterior density is not available, we employ a numerical grid approximation technique to maintain and update the posterior distribution of $\theta$ (stiffness efficiency factor) over its bounded domain. This involves discretizing the $\theta$ space and sequentially updating the probability mass/density at each grid point.

### Step-by-Step Numerical Procedure:

1.  **Define the $\theta$-Grid and Boundary Limits:**
    *   **Domain:** The physical domain for $\theta$ is $(0, 1]$. We need to select a suitable range for our grid, e.g., $\theta_{min}$ (slightly greater than 0) and $\theta_{max}$ (equal to 1). A common choice for $\theta_{min}$ would be a small positive number like 0.001 or 0.01 to avoid issues with $\ln(\theta)$ approaching negative infinity. For this problem, we'll use $\theta_{min} = 0.01$ and $\theta_{max} = 1.0$.
    *   **Discretization:** Divide this range into a fixed number of equally spaced points, say $M$ points: $\theta_{grid} = \{\theta_1, \theta_2, \dots, \theta_M\}$.
    *   **Grid Spacing:** Calculate the step size $\Delta\theta = (\theta_{max} - \theta_{min}) / (M - 1)$. This $\Delta\theta$ is crucial for numerical integration (e.g., trapezoidal rule).

2.  **Initialize the Prior Distribution (Step $k=0$):**
    *   **Evaluate Prior Density:** For each point $\theta_j$ on the grid, evaluate the initial Beta prior density $f_{\Theta}^{(0)}(\theta_j) = \text{Beta}(\theta_j \mid \alpha_0, \beta_0)$. Let's denote this as $P_{prior}^{(0)}[j]$.
    *   **Numerical Normalization (Initial):** For a discrete grid representation of a continuous PDF, the sum of the densities multiplied by the grid spacing should integrate to 1. Using the trapezoidal rule for the initial normalization:
        $$Z_0 = \int_{\theta_{min}}^{\theta_{max}} f_{\Theta}^{(0)}(\theta) d\theta \approx \sum_{j=1}^{M} \left( \frac{P_{prior}^{(0)}[j] + P_{prior}^{(0)}[j+1]}{2} \right) \Delta\theta \quad \text{ (or simply } \text{np.trapz(P}_{prior}^{(0)}, \theta_{grid}))$$
        The initial normalized posterior (which becomes the prior for $k=1$) is then:
        $$P_{normalized}^{(0)}[j] = P_{prior}^{(0)}[j] / Z_0$$
        This $P_{normalized}^{(0)}[j]$ represents the discrete approximation of the posterior PDF after 0 observations.

3.  **Sequential Update for each new sensor reading $y_k$ (for $k = 1, 2, \dots, n$):**
    *   **Prior for current step:** The normalized posterior from the previous step becomes the prior for the current step: $P_{prior}^{(k)}[j] = P_{normalized}^{(k-1)}[j]$.
    *   **Calculate Likelihood Contribution:** For the current sensor reading $y_k$ and known parameters $K_{\text{nominal}}$, $\sigma$:
        For each $\theta_j$ on the grid, calculate the likelihood $L(y_k \mid \theta_j)$ using the log-normal PDF formula from Task 2:
        $$L(y_k \mid \theta_j) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left(-\frac{\left(\ln\left(\frac{y_k}{\theta_j \cdot K_{\text{nominal}}}\right)\right)^2}{2\sigma^2}\right)$$
    *   **Update Unnormalized Posterior:** Multiply the likelihood contribution by the prior (from the previous step's normalized posterior) at each grid point:
        $$P_{unnormalized}^{(k)}[j] = L(y_k \mid \theta_j) \cdot P_{prior}^{(k)}[j]$$

4.  **Sequential Normalization Step (using Trapezoidal Rule):**
    *   After calculating the unnormalized posterior $P_{unnormalized}^{(k)}[j]$ for all grid points, the distribution must be normalized so that its integral over the domain $(0,1]$ equals 1. This step computes the evidence $P(y_k \mid \mathbf{y}^{(k-1)})$.
    *   Use the **trapezoidal rule** for numerical integration to find the normalizing constant $Z_k$:
        $$Z_k = \int_{\theta_{min}}^{\theta_{max}} P_{unnormalized}^{(k)}(\theta) d\theta \approx \text{np.trapz}(P_{unnormalized}^{(k)}, \theta_{grid})$$
    *   Normalize each grid point's value by this constant:
        $$P_{normalized}^{(k)}[j] = P_{unnormalized}^{(k)}[j] / Z_k$$
    *   This $P_{normalized}^{(k)}[j]$ now represents the approximate posterior probability density function for $\theta$ after observing $k$ sensor readings. It is stored and used as $P_{prior}^{(k+1)}[j]$ for the next sequential update.

## Task 6

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta, norm

# --- 1. Define Model Functions and Simulation Parameters ---

# Likelihood function for a single sensor measurement y_k
def likelihood_shm(y_k, theta, K_nominal, sigma):
    # Ensure theta is not zero to avoid log(0) issues
    theta = np.clip(theta, 1e-10, None)

    # Calculate the argument for the log-normal PDF exponent
    # ln(y_k / (theta * K_nominal)) simplifies ln(y_k) - ln(theta * K_nominal)
    log_term = np.log(y_k) - np.log(theta * K_nominal)

    # Log-normal PDF formula
    numerator = np.exp(-((log_term)**2) / (2 * sigma**2))
    denominator = y_k * sigma * np.sqrt(2 * np.pi)

    # Handle potential division by zero if y_k is extremely small (shouldn't happen with log-normal)
    return numerator / denominator

# Function to simulate a single noisy sensor reading y_k
def simulate_sensor_reading(theta_true, K_nominal, sigma):
    epsilon_k = np.random.normal(0, sigma) # Noise term
    y_k = theta_true * K_nominal * np.exp(epsilon_k)
    return y_k

# Simulation Parameters
theta_true = 0.68      # True, hidden remaining stiffness efficiency
n_measurements = 15    # Number of continuous sensor measurements
K_nominal = 50.0       # Nominal stiffness in kN/mm
sigma = 0.15           # Standard deviation of sensor noise in log-space

# Initial Prior Parameters (from Task 1)
alpha_0 = 8
beta_0 = 1.5

# --- 2. Define the θ-Grid ---
theta_min, theta_max = 0.01, 1.0 # Bounded physical domain for theta
n_grid_points = 500              # Number of grid points for approximation
theta_grid = np.linspace(theta_min, theta_max, n_grid_points)

# --- 3. Initialize the Prior Distribution ---
# Evaluate the initial Beta prior density on the grid
initial_prior_density = beta.pdf(theta_grid, alpha_0, beta_0)

# Normalize the initial prior using trapezoidal rule (area under PDF should be 1)
Z_0 = np.trapz(initial_prior_density, theta_grid)
current_posterior_density = initial_prior_density / Z_0

# --- 4. Store History of Estimators and Posterior Curves ---
posterior_mean_history = []
map_estimate_history = []
posterior_curve_milestones = {0: current_posterior_density.copy()}

# --- Step 0: Initial estimates (before any sensor readings) ---
# Posterior Mean for initial prior
initial_posterior_mean = np.trapz(theta_grid * current_posterior_density, theta_grid)
posterior_mean_history.append(initial_posterior_mean)

# MAP for initial prior (mode of Beta(8, 1.5) = (8-1)/(8+1.5-2) = 7/7.5 ~ 0.933)
# For a Beta(alpha, beta) with alpha > 1, beta > 1, mode is (alpha-1)/(alpha+beta-2)
if alpha_0 > 1 and beta_0 > 1:
    initial_map_estimate = (alpha_0 - 1) / (alpha_0 + beta_0 - 2)
else:
    initial_map_estimate = theta_grid[np.argmax(current_posterior_density)]
map_estimate_history.append(initial_map_estimate)

# --- 5. Simulation Loop for Sequential Updates ---
for k in range(1, n_measurements + 1):
    # Simulate a new sensor reading y_k based on theta_true
    y_k = simulate_sensor_reading(theta_true, K_nominal, sigma)

    # Calculate likelihood contribution for the current y_k across the theta_grid
    likelihood_k = likelihood_shm(y_k, theta_grid, K_nominal, sigma)

    # Update unnormalized posterior: Likelihood * Prior (from previous step's normalized posterior)
    unnormalized_posterior = likelihood_k * current_posterior_density

    # Sequential Normalization Step using np.trapz
    Z_k = np.trapz(unnormalized_posterior, theta_grid)
    current_posterior_density = unnormalized_posterior / Z_k

    # Calculate and store estimators
    # Posterior Mean (numerical integration using trapezoidal rule)
    posterior_mean = np.trapz(theta_grid * current_posterior_density, theta_grid)
    posterior_mean_history.append(posterior_mean)

    # MAP Estimate (value on grid corresponding to max posterior density)
    map_estimate = theta_grid[np.argmax(current_posterior_density)]
    map_estimate_history.append(map_estimate)

    # Store posterior curve at specified milestones
    if k in [1, 2, 5, 10, 15]:
        posterior_curve_milestones[k] = current_posterior_density.copy()

# ------------------ 6. Visualization ------------------

# Plot 1: Progression of full posterior density curves at milestones
fig1 = go.Figure()

for k_val, density_curve in posterior_curve_milestones.items():
    fig1.add_trace(go.Scatter(
        x=theta_grid,
        y=density_curve,
        mode='lines',
        name=f'Posterior at k={k_val}'
    ))

# Add true theta_true as a vertical line
fig1.add_vline(x=theta_true, line_dash='dash', line_color='red',
               annotation_text=f'True θ = {theta_true}', annotation_position='top right')

fig1.update_layout(
    title='Progression of Posterior Density Curves Over Time',
    xaxis_title='Stiffness Efficiency Factor (θ)',
    yaxis_title='Posterior Probability Density',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)', bordercolor='rgba(0,0,0,0.5)', borderwidth=1)
)
fig1.show()

# Plot 2: Line chart tracking convergence of Posterior Mean and MAP
fig2 = go.Figure()

# Add Posterior Mean trace
fig2.add_trace(go.Scatter(
    x=list(range(n_measurements + 1)),
    y=posterior_mean_history,
    mode='lines+markers',
    name='Posterior Mean',
    marker=dict(symbol='circle', size=7)
))

# Add MAP Estimate trace
fig2.add_trace(go.Scatter(
    x=list(range(n_measurements + 1)),
    y=map_estimate_history,
    mode='lines+markers',
    name='MAP Estimate',
    marker=dict(symbol='square', size=7)
))

# Add true theta reference line
fig2.add_hline(y=theta_true, line_dash='dash', line_color='red',
              annotation_text=f'True θ = {theta_true}', annotation_position='bottom right')

fig2.update_layout(
    title='Convergence of Stiffness Estimators Over Sensor Readings',
    xaxis_title='Number of Sensor Readings (k)',
    yaxis_title='Estimated Stiffness Efficiency (θ)',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)', bordercolor='rgba(0,0,0,0.5)', borderwidth=1)
)
fig2.show()

/tmp/ipykernel_2316/2616744588.py:49: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2316/2616744588.py:59: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2316/2616744588.py:82: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2316/2616744588.py:87: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



### Analysis:

**Behavior of the Posterior Distribution and Estimators:**

The first plot, "Progression of Posterior Density Curves Over Time," visually demonstrates how the model's belief about the stiffness efficiency factor (θ) evolves with each new sensor reading.

*   **Initial State (k=0):** The initial posterior curve (which is the prior Beta(8, 1.5)) is highly concentrated around $\theta \approx 0.93$, reflecting the optimistic belief that the component is largely healthy. The initial Posterior Mean and MAP are high, consistent with this prior.

*   **Early Readings (k=1, 2, 5):** With the first few sensor readings, which are simulated around $\theta_{\text{true}}=0.68$, the posterior distribution starts shifting its mass away from the initial optimistic peak. The density curve gradually moves to the left, and its peak (MAP) and mean (Posterior Mean) begin to decrease. The distribution also starts to widen slightly as it tries to reconcile the optimistic prior with incoming data indicating a lower stiffness.

*   **Convergence (k=10, 15):** As more sensor readings accumulate (e.g., at k=10 and k=15), the posterior density curve becomes increasingly centered and concentrated around $\theta_{\text{true}}=0.68$. The optimistic prior's influence diminishes significantly. The curve becomes taller and narrower, indicating increased certainty in the estimate.

**Overcoming the Optimistic Prior:**

The second plot, "Convergence of Stiffness Estimators Over Sensor Readings," tracks the Posterior Mean and MAP estimates directly. Both estimators start high (around 0.93) due to the strong prior. It takes approximately **between 5 and 10 sensor readings** for both the Posterior Mean and MAP to confidently move away from the prior's peak and begin converging closely towards the true degradation state of $\theta_{\text{true}}=0.68$. By $k=15$, both estimators are very close to the true value, and their fluctuations have significantly reduced.

**Implication of Narrowing Density Curves:**

The narrowing of the posterior density curves over time (as seen in the first plot) implies a **reduction in uncertainty** about the true stiffness efficiency factor $\theta$. As the system accumulates more evidence from sensor readings, its confidence in the estimated degradation state increases.

For structural safety thresholds, this narrowing is crucial:

*   **Improved Decision-Making:** A narrower distribution means a more precise estimate of $\theta$. This allows engineers to make more informed decisions about the structural integrity. If the estimated $\theta$ (and its credible interval, which would be narrower with a sharper posterior) falls below a predefined safety threshold, timely maintenance or repair can be scheduled.

*   **Risk Assessment:** The spread of the posterior distribution quantifies the remaining uncertainty. A wide distribution means high uncertainty, suggesting a higher risk if the estimated mean is near a critical threshold. A narrow distribution, even if its mean is below a threshold, indicates a high probability that the component is indeed degraded, prompting action.

In essence, the Bayesian tracking system provides a dynamic, evidence-based assessment of structural health, allowing for early detection of degradation and proactive safety measures.

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

### 1. Deriving the Marginal Density:

We are given the following:
*   Prior probability of cluster membership: $P(C_i=k)=\phi_k$
*   Conditional density of $X_i$ given $C_i=k$: $X_i\mid C_i=k \sim \mathscr N(\mu_k,\Sigma_k)$, meaning its probability density function is $p(x_i\mid C_i=k) = \mathscr N(x_i\mid \mu_k,\Sigma_k)$.

To derive the marginal density of $X_i$, $p(x_i)$, we use the law of total probability, which states that for a continuous variable $X$ and a discrete variable $C$, $p(x) = \sum_{k} p(x \mid C=k)P(C=k)$.

Applying this to our specific case:

$$p(x_i) = \sum_{k=1}^K p(x_i \mid C_i=k)P(C_i=k)$$

Substitute the given expressions for $p(x_i\mid C_i=k)$ and $P(C_i=k)$:

$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr N(x_i\mid \mu_k,\Sigma_k)$$

This is precisely the desired marginal density of $X_i$.

### Explanation of "Gaussian mixture density":

This density is called a **Gaussian mixture density** because it represents a weighted sum of several Gaussian (normal) probability density functions. Each component $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ is a Gaussian density, and these densities are "mixed" together with weights $\phi_k$. The weights $\phi_k$ are the prior probabilities of belonging to each cluster, and they sum to 1 ($\sum_{k=1}^K \phi_k=1$), making them valid mixing coefficients.

Essentially, the model assumes that each observation $x_i$ is generated by first choosing one of $K$ latent (hidden) Gaussian distributions with probability $\phi_k$, and then drawing $x_i$ from that chosen Gaussian distribution. The overall density $p(x_i)$ is thus a composite of these individual Gaussian components.

### 2. Deriving the Posterior Cluster Probability:

For a fixed observation $x_i$, we use Bayes' rule to derive the posterior probability of cluster membership $P(C_i=k\mid X_i=x_i)$.

Bayes' Rule states:

$$P(C_i=k\mid X_i=x_i)=\frac{P(X_i=x_i\mid C_i=k)P(C_i=k)}{P(X_i=x_i)}$$

From **Part 1**, we know that the marginal density $P(X_i=x_i)$ is given by the law of total probability:

$$P(X_i=x_i) = \sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)$$

Substituting this into Bayes' Rule, we get the initial form:

$$P(C_i=k\mid X_i=x_i)=\frac{P(X_i=x_i\mid C_i=k)P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)}$$

Now, we substitute the Gaussian model and the cluster prior:
*   $P(X_i=x_i\mid C_i=k) = \mathscr N(x_i\mid \mu_k,\Sigma_k)$
*   $P(C_i=k)=\phi_k$

Substituting these into the equation:

$$P(C_i=k\mid X_i=x_i)=\frac{\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)}{\sum_{j=1}^K \phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)}$$

This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by $\gamma_{ik}$.

### Explanation of why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership:

$\gamma_{ik}$ is the **posterior probability of cluster membership** because it represents our updated belief about the probability that observation $x_i$ belongs to cluster $k$, *after* we have observed the data point $x_i$.

*   **Prior Information:** $P(C_i=k) = \phi_k$ is the *prior probability* that any given observation belongs to cluster $k$, before we have seen any specific data point $x_i$. It reflects the overall proportion of data points expected to be in cluster $k$.
*   **Likelihood:** $P(X_i=x_i\mid C_i=k) = \mathscr N(x_i\mid \mu_k,\Sigma_k)$ is the likelihood of observing $x_i$ if it were generated by cluster $k$. It measures how well data point $x_i$ 'fits' or is 'compatible' with the characteristics (mean and covariance) of cluster $k$.
*   **Evidence (Marginal Likelihood):** The denominator $\sum_{j=1}^K \phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)$ is the marginal probability of observing $x_i$, effectively averaging the likelihoods over all possible cluster memberships, weighted by their prior probabilities. It acts as a normalizing constant.

By combining the prior belief about cluster sizes ($\phi_k$) with the likelihood of $x_i$ given each cluster ($\mathscr N(x_i\mid \mu_k,\Sigma_k)$), Bayes' rule updates our initial belief. Thus, $\gamma_{ik}$ provides the probabilistic assignment of $x_i$ to cluster $k$ that is consistent with both our prior expectations and the observed data. It's 'posterior' because it comes *after* considering the evidence $x_i$.

### 3. One-Hot Encoding of the Latent Cluster Variable:

We define a one-hot encoded latent random vector:

$$Z_i=\begin{bmatrix}Z_{i1}\\Z_{i2}\\\;\;\vdots\\Z_{iK}\end{bmatrix},$$

where

$$Z_{ik}=\begin{cases}1, & \text{if } C_i=k,\\0, & \text{otherwise}.\end{cases}$$

#### Show that $\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i)$:

By definition, the expectation of a discrete random variable is the sum of each possible value multiplied by its probability. For $Z_{ik}$, it can take two values: 1 (if $C_i=k$) or 0 (if $C_i \ne k$).

Therefore, the conditional expectation of $Z_{ik}$ given $X_i=x_i$ is:

$$\mathbb E[Z_{ik}\mid X_i=x_i] = (1) \cdot P(Z_{ik}=1\mid X_i=x_i) + (0) \cdot P(Z_{ik}=0\mid X_i=x_i)$$

Since $Z_{ik}=1$ if and only if $C_i=k$, we can substitute $P(Z_{ik}=1\mid X_i=x_i)$ with $P(C_i=k\mid X_i=x_i)$:

$$\mathbb E[Z_{ik}\mid X_i=x_i] = P(C_i=k\mid X_i=x_i)$$

This shows the desired relationship.

#### Hence show that $\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}\gamma_{i1}\\\;\gamma_{i2}\\\;\vdots\\\gamma_{iK}\end{bmatrix}$:

We know that $\gamma_{ik} = P(C_i=k\mid X_i=x_i)$ from **Part 2**. Using the result we just derived, we can write:

$$\mathbb E[Z_{ik}\mid X_i=x_i] = \gamma_{ik}$$

The vector $\mathbb E[Z_i\mid X_i=x_i]$ is simply the vector of the conditional expectations of its components:

$$\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}\mathbb E[Z_{i1}\mid X_i=x_i]\\\mathbb E[Z_{i2}\mid X_i=x_i]\\\vdots\\\mathbb E[Z_{iK}\mid X_i=x_i]\end{bmatrix}$$

Substituting $\mathbb E[Z_{ik}\mid X_i=x_i] = \gamma_{ik}$ into the vector:

$$\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}\gamma_{i1}\\ \gamma_{i2}\\\;\vdots\\\gamma_{iK}\end{bmatrix}$$

#### Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation $\mathbb E[Z_i\mid X_i=x_i]$:

The vector $\mathbb E[Z_i\mid X_i=x_i]$ contains the probabilities that observation $x_i$ belongs to each of the $K$ clusters, given $x_i$. These probabilities are the responsibilities $\gamma_{ik}$. When we assign $x_i$ to clusters based on these probabilities rather than a single, definitive cluster, we are performing a "soft assignment." Each element $\gamma_{ik}$ indicates the degree or "softness" of assignment of $x_i$ to cluster $k$. Therefore, the soft cluster assignment in a Gaussian Mixture Model is precisely given by the conditional expectation $\mathbb E[Z_i\mid X_i=x_i]$.

### 4. From Soft Assignment to Hard Clustering:

The vector
$$\mathbb E[Z_i\mid X_i=x_i] = \begin{bmatrix}\gamma_{i1}\\ \gamma_{i2}\\\;\vdots\\\gamma_{iK}\end{bmatrix}$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:

$$\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.$$

### Explain the difference between soft clustering and hard clustering in this context:

In the context of Gaussian Mixture Models:

*   **Soft Clustering (Probabilistic Assignment):** This is represented by the responsibilities $\gamma_{ik}$ (and thus by the conditional expectation vector $\mathbb E[Z_i\mid X_i=x_i]$). In soft clustering, each data point $x_i$ is not exclusively assigned to a single cluster. Instead, it is assigned a *probability* (or a degree of membership) for *each* of the $K$ clusters. The values $\gamma_{ik}$ for a given $x_i$ sum to 1 across all $k$ clusters, indicating the model's belief about the likelihood of $x_i$ belonging to each cluster. This provides a richer, more nuanced understanding of cluster membership, especially for data points that lie in ambiguous regions between clusters. For such points, multiple $\gamma_{ik}$ values might be relatively high, reflecting uncertainty in their assignment.

*   **Hard Clustering (Deterministic Assignment):** This approach forces each data point $x_i$ to belong to exactly one cluster. In the GMM framework, a hard assignment is derived from the soft assignments by simply selecting the cluster $k$ for which $x_i$ has the highest responsibility $\gamma_{ik}$. That is, $x_i$ is assigned to cluster $\widehat C_i = \operatorname{arg\,max}_{1\le k\le K} \gamma_{ik}$. This reduces the information about membership to a single, definitive label, discarding the degrees of membership provided by the responsibilities. While simpler and often easier to interpret, hard clustering loses information about the model's uncertainty for individual data points.

**In essence, the difference lies in the output:** Soft clustering gives a vector of probabilities for each data point, indicating how strongly it belongs to each cluster, while hard clustering gives a single, definitive cluster label to each data point.

### 5. Conditional Expectation of the Observation Given the Cluster:

#### Show that $\mathbb E[X_i\mid C_i=k]=\mu_k$:

We are given that conditional on $C_i=k$, the observation $X_i$ is generated from a multivariate Gaussian distribution:

$$X_i\mid C_i=k \sim \mathscr N(\mu_k,\Sigma_k).$$

By definition, the mean (or expected value) of a multivariate Gaussian distribution $\mathscr N(\mu, \Sigma)$ is its mean vector $\mu$.

Therefore, directly from the definition of the conditional distribution:

$$\mathbb E[X_i\mid C_i=k]=\mu_k.$$

#### Explain why $\mu_k$ can be interpreted as the center of cluster $k$:

The parameter $\mu_k$ is the mean vector of the $k$-th Gaussian component. In the context of clustering, if we were to sample an infinite number of data points *only* from cluster $k$, their average value would converge to $\mu_k$. Thus, $\mu_k$ represents the **central tendency** or the **average location** of the data points belonging to cluster $k$. It acts as the 'centroid' or 'center' of that particular cluster in the feature space. Data points within cluster $k$ are expected to be distributed around this mean vector.

#### Compare the two conditional expectations $\mathbb E[Z_i\mid X_i=x_i]$ and $\mathbb E[X_i\mid C_i=k]$:

We have two distinct conditional expectations, each providing different types of information:

1.  **$\mathbb E[Z_i\mid X_i=x_i] = [\gamma_{i1}, \dots, \gamma_{iK}]^T$:** This vector, derived in **Part 3**, represents the conditional expectation of the **latent cluster membership variable $Z_i$** given an observed data point $X_i=x_i$. Each element $\gamma_{ik}$ is the posterior probability that $x_i$ belongs to cluster $k$. This vector quantifies *where the observed data point $x_i$ likely belongs among the clusters*.

2.  **$\mathbb E[X_i\mid C_i=k] = \mu_k$:** This vector represents the conditional expectation of the **observed data point $X_i$** given that it belongs to a specific cluster $C_i=k$. This is simply the mean vector of cluster $k$. This vector quantifies *where the center of cluster $k$ is located in the feature space*.

#### Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster:

*   **$\mathbb E[Z_i\mid X_i=x_i]$ gives the soft cluster membership of an observed point:** This expectation tells us, for a *given observed data point* $x_i$, what is the probability that it came from each of the $K$ clusters. It's a vector of probabilities (responsibilities) that sum to 1, indicating the *degree of belonging* of $x_i$ to each cluster. This is precisely what we mean by "soft cluster membership" – $x_i$ is not definitively assigned to one cluster but rather has a probabilistic association with all of them.

*   **$\mathbb E[X_i\mid C_i=k]$ gives the mean location of a cluster:** This expectation tells us, *given that a data point belongs to cluster $k$*, what its expected feature values are. It is the definition of the cluster's mean vector, $\mu_k$. It describes the *typical value* of the observations that belong to cluster $k$, effectively defining its spatial location or center in the feature space. It doesn't tell us about the membership of a specific $x_i$, but rather about the characteristics of the cluster itself.

### 6. The Complete-Data Likelihood

If the latent labels $Z_i$ were known, the complete-data likelihood would be

$$p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.$$

#### Take the logarithm and show that the complete-data log-likelihood is:

$$\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].$$

Let's start by taking the natural logarithm of the complete-data likelihood:

$$\ell_c = \log\left(\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}\right)$$

Using the logarithm property $\log(AB) = \log A + \log B$ and $\log(A^B) = B \log A$, we can bring the products outside as sums:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K \log\left( \left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}} \right)$$

Now, apply the power rule for logarithms:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \log\left( \phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k) \right)$$

Finally, apply the product rule for logarithms inside the brackets:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr N(x_i\mid \mu_k,\Sigma_k) \right]$$

This matches the desired expression for the complete-data log-likelihood.

#### Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known:

If the values of $z_{ik}$ were known, this expression would be easy to maximize with respect to the model parameters $(\phi_k, \mu_k, \Sigma_k)$ because:

1.  **Decoupling of Parameters:** The log-likelihood separates into independent terms for each cluster $k$ and each observation $i$, weighted by $z_{ik}$. Critically, the parameters for the mixing proportions ($\phi_k$), means ($\mu_k$), and covariances ($\Sigma_k$) become largely decoupled. We can maximize each set of parameters independently.

2.  **Explicit Cluster Assignment:** Each $z_{ik}$ is either 0 or 1. If $z_{ik}=1$, it means observation $x_i$ definitively belongs to cluster $k$. If $z_{ik}=0$, it definitively does not. This effectively turns the problem into a standard maximum likelihood estimation for each cluster separately, where each observation $x_i$ is explicitly assigned to a single cluster.

    *   For example, to find the new $\phi_k$ (mixing proportions), we would simply count the number of data points assigned to cluster $k$ (i.e., $\sum_{i=1}^n z_{ik}$) and divide by the total number of data points $n$. This is a straightforward counting problem.

    *   To find $\mu_k$ and $\Sigma_k$ for a given cluster $k$, we would only consider the data points for which $z_{ik}=1$ (i.e., data points assigned to cluster $k$). The maximization problems for $\mu_k$ and $\Sigma_k$ then reduce to the standard maximum likelihood estimates for a single Gaussian distribution, which have well-known closed-form solutions (the sample mean and sample covariance of the data points assigned to that cluster).

3.  **Absence of Sum in Logarithm:** In the incomplete-data log-likelihood (which is $\sum_{i=1}^n \log \left( \sum_{k=1}^K \phi_k \mathscr N(x_i\mid \mu_k,\Sigma_k) \right)$), the logarithm is applied *after* the sum over $k$, making it analytically intractable to maximize. However, in the complete-data log-likelihood, the sum over $k$ is *outside* the logarithm, and each term is weighted by $z_{ik}$. This structure allows for much simpler differentiation and analytical maximization.

### 7. The EM Interpretation:

In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:

$$z_{ik} \quad\leadsto\quad \mathbb E[Z_{ik}\,|\,X_i=x_i].$$

That is,

$$z_{ik} \quad\leadsto\quad \gamma_{ik}.$$

This is the E-step of the EM algorithm. Using this idea, we write the expected complete-data log-likelihood:

$$Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].$$

### Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities:

The E-step (Expectation step) in the EM algorithm for Gaussian Mixture Models is fundamentally about **computing the expected values of the latent variables $Z_{ik}$ given the observed data $X_i$ and the current estimates of the model parameters $(\phi_k, \mu_k, \Sigma_k)$**. As we showed in Part 3, these conditional expectations are precisely the responsibilities $\gamma_{ik} = P(C_i=k\mid X_i=x_i)$.

Therefore, the E-step can be interpreted as a **conditional update of cluster membership probabilities** because:

1.  **Probabilistic Assignment:** For each data point $x_i$, the E-step calculates a set of probabilities $(\gamma_{i1}, \dots, \gamma_{iK})$. These are not fixed assignments but rather probabilistic beliefs about how likely $x_i$ is to belong to each cluster $k$, given the current state of the model parameters.

2.  **"Soft" Membership:** Instead of a "hard" assignment where $x_i$ either belongs entirely to cluster $k$ ($z_{ik}=1$) or not ($z_{ik}=0$), the E-step provides a "soft" assignment. The responsibility $\gamma_{ik}$ represents the fractional membership of $x_i$ in cluster $k$. For example, if $\gamma_{i1}=0.7$ and $\gamma_{i2}=0.3$, then $x_i$ is considered to be 70% in cluster 1 and 30% in cluster 2.

3.  **Conditional on Current Parameters:** The calculation of $\gamma_{ik}$ depends directly on the current estimates of $\phi_k$, $\mu_k$, and $\Sigma_k$. As these parameters are refined in subsequent M-steps, the calculated responsibilities will also be updated, reflecting a continuously evolving understanding of cluster memberships.

4.  **Information Extraction:** The E-step effectively uses the current model to extract as much information as possible about the unobserved latent variables (cluster memberships) from the observed data. These extracted membership probabilities ($\gamma_{ik}$) then serve as "pseudo-labels" or "weighted counts" for the subsequent M-step, allowing the parameters to be updated as if the true labels were known (but in a weighted, fractional sense).

### 8. Parameter Updates:

We need to maximize the expected complete-data log-likelihood $Q$ with respect to the model parameters $(\phi_k, \mu_k, \Sigma_k)$. Recall the expression for $Q$:

$$Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].$$

#### Deriving the update for $\phi_k^{\text{new}}$ (Mixing Proportions):

To maximize $Q$ with respect to $\phi_k$, we only consider the terms involving $\log \phi_k$ and introduce a Lagrange multiplier $\lambda$ to enforce the constraint $\sum_{k=1}^K \phi_k = 1$. (Also, $\phi_k \ge 0$ is implicitly handled by the nature of the solution).

$$Q_{\phi} = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k + \lambda\left(\sum_{k=1}^K \phi_k - 1\right)$$

Take the derivative with respect to $\phi_k$ and set it to zero:

$$\frac{\partial Q_{\phi}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} + \lambda = 0$$

$$\sum_{i=1}^n \gamma_{ik} = -\lambda \phi_k$$

Summing over $k$:

$$\sum_{k=1}^K \sum_{i=1}^n \gamma_{ik} = -\lambda \sum_{k=1}^K \phi_k$$

Since $\sum_{k=1}^K \phi_k = 1$, we get:

$$\sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} = -\lambda$$

We know that $\sum_{k=1}^K \gamma_{ik} = \sum_{k=1}^K P(C_i=k\mid X_i=x_i) = 1$ (because for each $x_i$, the responsibilities sum to 1 over all clusters). So, $\sum_{i=1}^n 1 = n$.

Therefore, $-\lambda = n$, or $\lambda = -n$.

Substitute $\lambda = -n$ back into $\sum_{i=1}^n \gamma_{ik} = -\lambda \phi_k$:

$$\sum_{i=1}^n \gamma_{ik} = n \phi_k$$

Let $N_k = \sum_{i=1}^n \gamma_{ik}$, which is the effective number of points assigned to cluster $k$. Then:

$$N_k = n \phi_k^{\text{new}}$$

$$\phi_k^{\text{new}}=\frac{N_k}{n}$$

#### Deriving the update for $\mu_k^{\text{new}}$ (Mean Vector):

To maximize $Q$ with respect to $\mu_k$, we only consider terms involving $\mu_k$. The $\log \mathscr N(x_i\mid \mu_k,\Sigma_k)$ term is:

$$\log \mathscr N(x_i\mid \mu_k,\Sigma_k) = -\frac{d}{2}\log(2\pi) - \frac{1}{2}\log|\Sigma_k| - \frac{1}{2}(x_i-\mu_k)^T\Sigma_k^{-1}(x_i-\mu_k)$$

We need to maximize $Q_{\mu_k} = \sum_{i=1}^n \gamma_{ik} \left[ - \frac{1}{2}(x_i-\mu_k)^T\Sigma_k^{-1}(x_i-\mu_k) \right]$ with respect to $\mu_k$. This is equivalent to minimizing $\sum_{i=1}^n \gamma_{ik} (x_i-\mu_k)^T\Sigma_k^{-1}(x_i-\mu_k)$.

Take the derivative with respect to $\mu_k$ and set it to zero:

$$\frac{\partial}{\partial \mu_k} \left[ -\frac{1}{2} \sum_{i=1}^n \gamma_{ik} (x_i-\mu_k)^T\Sigma_k^{-1}(x_i-\mu_k) \right] = 0$$

$$\sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1}(x_i-\mu_k) = 0$$

Since $\Sigma_k^{-1}$ is invertible, we can multiply by $\Sigma_k$ to remove it:

$$\sum_{i=1}^n \gamma_{ik} (x_i-\mu_k) = 0$$

$$\sum_{i=1}^n \gamma_{ik} x_i - \sum_{i=1}^n \gamma_{ik} \mu_k = 0$$

$$\sum_{i=1}^n \gamma_{ik} x_i = \mu_k \sum_{i=1}^n \gamma_{ik}$$

Using $N_k = \sum_{i=1}^n \gamma_{ik}$:

$$\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i$$

#### Deriving the update for $\Sigma_k^{\text{new}}$ (Covariance Matrix):

To maximize $Q$ with respect to $\Sigma_k$, we consider terms involving $\Sigma_k$:

$$Q_{\Sigma_k} = \sum_{i=1}^n \gamma_{ik} \left[ -\frac{1}{2}\log|\Sigma_k| - \frac{1}{2}(x_i-\mu_k)^T\Sigma_k^{-1}(x_i-\mu_k) \right]$$

We know that $\frac{\partial}{\partial A} \log|A| = (A^{-1})^T$ and $\frac{\partial}{\partial A} x^T A^{-1} y = -A^{-T} x y^T A^{-T}$. For a symmetric matrix $\Sigma_k$, this simplifies. Setting the derivative to zero:

$$-\frac{1}{2}N_k\Sigma_k^{-1} + \frac{1}{2}\sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1}(x_i-\mu_k)(x_i-\mu_k)^T\Sigma_k^{-1} = 0$$

Multiply by $2\Sigma_k$ and rearrange:

$$N_k \Sigma_k = \sum_{i=1}^n \gamma_{ik} (x_i-\mu_k)(x_i-\mu_k)^T$$

$$\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}}) (x_i-\mu_k^{\text{new}})^T$$

Note that we use $\mu_k^{\text{new}}$ in the formula for $\Sigma_k^{\text{new}}$.

These are the standard GMM parameter updates.

### Explanation of how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$:

The responsibility $\gamma_{ik} = P(C_i=k\mid X_i=x_i)$ quantifies the posterior probability that observation $x_i$ belongs to cluster $k$. In the M-step, these responsibilities are used to update the model parameters. Instead of making a hard assignment (where $x_i$ either fully belongs to a cluster or not), the EM algorithm uses $\gamma_{ik}$ as a **fractional weight** for $x_i$'s contribution to cluster $k$'s parameters.

*   **For $\phi_k^{\text{new}}$:** The sum $\sum_{i=1}^n \gamma_{ik}$ ($N_k$) represents the *effective number of data points* assigned to cluster $k$. If $x_i$ has a high $\gamma_{ik}$ (e.g., 0.9), it contributes 0.9 to the count for cluster $k$. If it has a low $\gamma_{ik}$ (e.g., 0.1), it contributes 0.1. This effectively smooths the counting process, allowing points to partially contribute to multiple clusters. $\phi_k^{\text{new}}$ is then the proportion of these effective points out of the total $n$.

*   **For $\mu_k^{\text{new}}$:** The new mean for cluster $k$ is a weighted average of all data points $x_i$. Each $x_i$ is weighted by its responsibility $\gamma_{ik}$ for cluster $k$. If $x_i$ is very likely to belong to cluster $k$, it has a large weight in the calculation of $\mu_k^{\text{new}}$, pulling the mean towards itself. If $x_i$ is unlikely to belong to cluster $k$, it has a small weight.

*   **For $\Sigma_k^{\text{new}}$:** Similarly, the new covariance matrix for cluster $k$ is a weighted average of the outer products $(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^T$, where each outer product is weighted by $\gamma_{ik}$. This ensures that data points more likely to belong to cluster $k$ have a greater influence on shaping its estimated covariance.

In essence, $\gamma_{ik}$ allows each data point to contribute to the updates of *all* clusters in proportion to its probabilistic membership in each, thereby enabling a

##Interpretation:

Gaussian Mixture Model (GMM) clustering can be viewed as an iterative process of conditional updating, elegantly handled by the Expectation-Maximization (EM) algorithm. It begins with an initial set of parameters for each Gaussian component: the mixture weight $\phi_k$$\phi_k$, which acts as the prior probability of cluster $k$$k$; the mean $\mu_k$$\mu_k$; and the covariance $\Sigma_k$$\Sigma_k$. The EM algorithm then alternates between two steps:

In the E-step (Expectation step), for each observation $x_i$$x_i$ and each cluster $k$$k$, we compute the responsibility $\gamma_{ik}$$\gamma_{ik}$. This $\gamma_{ik}$$\gamma_{ik}$ is the posterior probability that $x_i$$x_i$ belongs to cluster $k$$k$, given the observed data point $x_i$$x_i$ and the current model parameters. It is derived by considering how compatible $x_i$$x_i$ is with cluster $k$$k$, as measured by the Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$$\mathscr N(x_i\mid \mu_k,\Sigma_k)$, and weighting this by the cluster's prior probability $\phi_k$$\phi_k$. Essentially, the E-step calculates the soft assignment vector $\mathbb E[Z_i\mid X_i=x_i]$$\mathbb E[Z_i\mid X_i=x_i]$, which indicates the fractional membership of $x_i$$x_i$ in each cluster.

In the M-step (Maximization step), these calculated responsibilities $\gamma_{ik}$$\gamma_{ik}$ act as fractional weights. The model parameters (mixture weights $\phi_k$$\phi_k$, means $\mu_k$$\mu_k$, and covariances $\Sigma_k$$\Sigma_k$) are then updated to maximize the expected complete-data log-likelihood (Q-function). For instance, the new $\phi_k$$\phi_k$ is the sum of responsibilities for cluster $k$$k$ divided by the total number of data points, and the new $\mu_k$$\mu_k$ and $\Sigma_k$$\Sigma_k$ are weighted averages of the data points, where each $x_i$$x_i$ is weighted by its responsibility $\gamma_{ik}$$\gamma_{ik}$ for that particular cluster.

This process of conditionally estimating cluster memberships (E-step) and then updating cluster parameters based on these estimates (M-step) is repeated until convergence. Therefore, Gaussian mixture clustering is fundamentally a probabilistic clustering approach based on the conditional expectations of latent cluster membership variables, continually refining its understanding of both cluster assignments and cluster characteristics.

## Task 10:

In [14]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.gmm = GaussianMixture(n_components=n_components, random_state=random_state, covariance_type='full')
        self.scaler = StandardScaler()
        self.X_train = None
        self.X_test = None
        self.df = None
        self.features = None

    def load_and_prepare_data(self, url, features, test_size=0.2):
        self.df = pd.read_csv(url)
        self.features = features
        # Drop rows with NaN values in the selected features
        self.df.dropna(subset=self.features, inplace=True)

        X = self.df[self.features].values
        X_scaled = self.scaler.fit_transform(X)
        self.X_train, self.X_test = train_test_split(X_scaled, test_size=test_size, random_state=self.random_state)
        print(f"Data loaded with {len(self.df)} samples. Features: {features}")
        print(f"Training set size: {self.X_train.shape[0]}, Test set size: {self.X_test.shape[0]}")

    def fit_gmm(self):
        self.gmm.fit(self.X_train)
        print(f"GMM converged: {self.gmm.converged_}")
        print(f"Number of iterations: {self.gmm.n_iter_}")

    def evaluate_out_of_sample(self):
        log_likelihood_test = self.gmm.score(self.X_test)
        print(f"Average log-likelihood on test set: {log_likelihood_test:.4f}")
        return log_likelihood_test

    def _get_responsibilities_grid(self):
        # Determine the data range for the grid from the *entire* scaled dataset (train + test)
        X_combined_scaled = np.vstack((self.X_train, self.X_test))

        x_min, x_max = X_combined_scaled[:, 0].min() - 0.5, X_combined_scaled[:, 0].max() + 0.5
        y_min, y_max = X_combined_scaled[:, 1].min() - 0.5, X_combined_scaled[:, 1].max() + 0.5

        xx_scaled, yy_scaled = np.meshgrid(np.linspace(x_min, x_max, 200),
                                           np.linspace(y_min, y_max, 200))

        # Predict responsibilities for the scaled grid points
        responsibilities_grid = self.gmm.predict_proba(np.c_[xx_scaled.ravel(), yy_scaled.ravel()])
        # Find the cluster with the maximum responsibility for each grid point
        max_responsibilities_idx = np.argmax(responsibilities_grid, axis=1)
        # Reshape to match the grid
        max_responsibilities_idx = max_responsibilities_idx.reshape(xx_scaled.shape)

        # Inverse transform the grid points to original scale for plotting with unscaled data
        grid_points_scaled = np.c_[xx_scaled.ravel(), yy_scaled.ravel()]
        grid_points_unscaled = self.scaler.inverse_transform(grid_points_scaled)
        xx_unscaled = grid_points_unscaled[:, 0].reshape(xx_scaled.shape)
        yy_unscaled = grid_points_unscaled[:, 1].reshape(yy_scaled.shape)

        return xx_unscaled, yy_unscaled, max_responsibilities_idx

    def plot_2d_density_heatmap(self):
        fig = make_subplots(rows=2, cols=2,
                            column_widths=[0.8, 0.2], row_heights=[0.2, 0.8],
                            horizontal_spacing=0.02, vertical_spacing=0.02,
                            specs=[[{}, {'rowspan': 1, 'colspan': 1}],
                                   [{}, {}]])

        # Main 2D Heatmap
        hist_2d = go.Histogram2dContour(
            x=self.df[self.features[0]],
            y=self.df[self.features[1]],
            colorscale='Jet',
            reversescale=True,
            showscale=False,
            xaxis='x2', yaxis='y2'
        )
        fig.add_trace(hist_2d, row=2, col=1)

        # Marginal X-axis distribution
        hist_x = go.Histogram(
            x=self.df[self.features[0]],
            marker_color='lightblue',
            yaxis='y1',
            showlegend=False
        )
        fig.add_trace(hist_x, row=1, col=1)

        # Marginal Y-axis distribution
        hist_y = go.Histogram(
            y=self.df[self.features[1]],
            marker_color='lightblue',
            xaxis='x1',
            orientation='h',
            showlegend=False
        )
        fig.add_trace(hist_y, row=2, col=2)

        fig.update_layout(
            title_text=f'Empirical 2D Density Heatmap with Marginal Distributions ({self.features[0]} vs {self.features[1]})',
            xaxis = {'visible': False, 'matches': 'x2'},
            yaxis = {'visible': False, 'matches': 'y2'},
            xaxis2 = {'title': self.features[0]},
            yaxis2 = {'title': self.features[1]},
            yaxis1 = {'showticklabels': False},
            xaxis1 = {'showticklabels': False},
            hovermode='closest',
            height=600, width=800,
            template='plotly_white'
        )
        fig.show()

    def plot_training_assignment(self):
        xx_unscaled, yy_unscaled, max_responsibilities_idx = self._get_responsibilities_grid()

        # Original unscaled training data for plotting
        X_train_unscaled = self.scaler.inverse_transform(self.X_train)

        fig = go.Figure()

        # Contour map of maximum posterior responsibilities (using unscaled grid)
        fig.add_trace(go.Contour(
            x=xx_unscaled[0], # Use unscaled grid
            y=yy_unscaled[:, 0], # Use unscaled grid
            z=max_responsibilities_idx,
            colorscale=px.colors.qualitative.Plotly, # Use a qualitative colorscale for cluster regions
            showscale=False,
            name='Cluster Regions',
            contours_coloring='heatmap',
            opacity=0.4 # Make contour transparent to see points
        ))

        # Scatter plot of training data points, colored by GMM cluster assignments
        train_labels = self.gmm.predict(self.X_train)
        fig.add_trace(go.Scatter(
            x=X_train_unscaled[:, 0],
            y=X_train_unscaled[:, 1],
            mode='markers',
            name='Training Data Points',
            marker=dict(
                color=train_labels,
                colorscale=px.colors.qualitative.Plotly,
                line_width=1,
                line_color='black',
                size=5
            )
        ))

        fig.update_layout(
            title_text='Training Data: GMM Cluster Assignments with Responsibility Contours',
            xaxis_title=self.features[0],
            yaxis_title=self.features[1],
            height=600, width=800,
            template='plotly_white'
        )
        fig.show()

    def plot_test_assignment(self):
        xx_unscaled, yy_unscaled, max_responsibilities_idx = self._get_responsibilities_grid()

        # Original unscaled test data for plotting
        X_test_unscaled = self.scaler.inverse_transform(self.X_test)

        fig = go.Figure()

        # Contour map of maximum posterior responsibilities (using unscaled grid)
        fig.add_trace(go.Contour(
            x=xx_unscaled[0], # Use unscaled grid
            y=yy_unscaled[:, 0], # Use unscaled grid
            z=max_responsibilities_idx,
            colorscale=px.colors.qualitative.Plotly,
            showscale=False,
            name='Cluster Regions',
            contours_coloring='heatmap',
            opacity=0.4
        ))

        # Scatter plot of test data points, colored by GMM cluster assignments
        test_labels = self.gmm.predict(self.X_test)
        fig.add_trace(go.Scatter(
            x=X_test_unscaled[:, 0],
            y=X_test_unscaled[:, 1],
            mode='markers',
            name='Test Data Points',
            marker=dict(
                color=test_labels,
                colorscale=px.colors.qualitative.Plotly,
                line_width=1,
                line_color='black',
                size=5
            )
        ))

        fig.update_layout(
            title_text='Test Data: GMM Cluster Assignments with Responsibility Contours',
            xaxis_title=self.features[0],
            yaxis_title=self.features[1],
            height=600, width=800,
            template='plotly_white'
        )
        fig.show()

In [15]:
url = "https://www.kaggle.com/datasets/arjunbhasin2013/ccdata"

data_path = "/content/CC GENERAL.csv"
financial_features = ['PURCHASES', 'CREDIT_LIMIT'] # Example features

# Instantiate and run the GMMFinancialSegmenter
gmm_segmenter = GMMFinancialSegmenter(n_components=3, random_state=42)

# Load and prepare data
gmm_segmenter.load_and_prepare_data(data_path, financial_features)

# Fit GMM on training data
gmm_segmenter.fit_gmm()

# Evaluate out-of-sample performance
gmm_segmenter.evaluate_out_of_sample()

# Generate interactive visualizations
gmm_segmenter.plot_2d_density_heatmap()
gmm_segmenter.plot_training_assignment()
gmm_segmenter.plot_test_assignment()

Data loaded with 8949 samples. Features: ['PURCHASES', 'CREDIT_LIMIT']
Training set size: 7159, Test set size: 1790
GMM converged: True
Number of iterations: 19
Average log-likelihood on test set: -1.6465




Evaluation of the Plots for Task 10:
2D Density Heatmap with Marginal Distributions: This plot provides an excellent initial view of the data's structure. You can visually identify regions of higher data density, which often correspond to potential cluster centers. The marginal histograms help to understand the distribution of each feature independently.

Training Assignment Plot: This plot overlays the training data points on a continuous contour map.

Test Assignment Plot: Similar to the training plot, this one shows how unseen test data points fall within the defined cluster regions.

Visual Demonstration of the Soft Assignment Expectation Vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$$\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$:
In Part 3 of the theoretical assignment, you derived the responsibility $\gamma_{ik} = P(C_i=k \mid X_i=x_i)$$\gamma_{ik} = P(C_i=k \mid X_i=x_i)$ and showed that the soft assignment expectation vector for a given data point $x_i$$x_i$ is $\mathbb{E}[Z_i \mid X_i=x_i] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$$\mathbb{E}[Z_i \mid X_i=x_i] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$.

The continuous background contour maps in the "Training Assignment Plot" and "Test Assignment Plot" visually represent this concept in action for a grid of points (i.e., $x_{\text{grid}}$$x_{\text{grid}}$).

Each distinct colored region on the contour map corresponds to an area in the feature space where one particular cluster $k$$k$ has the highest responsibility (i.e., the highest posterior probability $\gamma_{ik}$$\gamma_{ik}$) among all $K$$K$ clusters. When the GMMFinancialSegmenter determines max_responsibilities_idx, it is essentially identifying, for each point on the grid, which cluster $k$$k$ has the largest $\gamma_{ik}$$\gamma_{ik}$.

The boundaries between these colored regions illustrate where the maximum responsibility shifts from one cluster to another. These are not sharp, hard boundaries but rather represent the decision boundaries where the model's confidence for assigning a point to one cluster becomes higher than for any other. They delineate the regions where $\mathbb{E}[Z_i \mid X_i=x_{\text{grid}}]$$\mathbb{E}[Z_i \mid X_i=x_{\text{grid}}]$ for a given $x_{\text{grid}}$$x_{\text{grid}}$ indicates that a specific cluster has the dominant posterior probability.

Points falling within a region are most likely to belong to the cluster associated with that color, according to the GMM's current parameters. However, even within a dominant region, a point still has non-zero (though lower) responsibilities for other clusters. The contour map, by showing discrete regions, visually simplifies this by displaying only the maximum responsibility, which is directly related to making a hard assignment from the soft responsibilities.

Therefore, the contour map effectively visualizes the outcome of comparing the elements of the soft assignment expectation vector for each point on the grid, thus showing which cluster is conditionally most probable at any given location in the feature space.

